# Analysis

Sentiment + categorization for discovery records.

Prereq: `python3 setup/extract_records.py` so
`output/processed/challenges.csv` and `expectations.csv` exist.

**Sentiment** — workplace zero-shot (`deberta-v3`) with:
- source priors (challenges → negative, expectations → positive)
- confidence-margin fallback
- pain / wish keyword overrides

**Categorize** — hybrid title keywords → body keywords → semantic (`all-mpnet-base-v2`)
with richer category blurbs and keyword↔semantic agreement boost.


In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import sys

import hdbscan
import numpy as np
import pandas as pd
import umap
from IPython.display import display
from sklearn.metrics import silhouette_score

# Local analysis helpers (survives notebook buffer overwrites)
sys.path.insert(0, str(Path(".").resolve()))
from setup.analyze_records import (
    CATEGORY_CONFIG,
    CATEGORY_DESCRIPTIONS,
    SENTIMENT_LABELS,
    SENTIMENT_MODEL,
    build_sentiment_classifier,
    realign_by_sentiment,
    resolve_sentiment_label,
    run_categorize,
    run_sentiment,
)

CHALLENGES_CSV = Path("./output/processed/challenges.csv")
EXPECTATIONS_CSV = Path("./output/processed/expectations.csv")
PROCESSED_DIR = Path("./output/processed")
CHALLENGES_SCORED = PROCESSED_DIR / "challenges_scored.csv"
EXPECTATIONS_SCORED = PROCESSED_DIR / "expectations_scored.csv"
CATEGORIZED_CHALLENGES_CSV = PROCESSED_DIR / "categorized_challenges.csv"
CATEGORIZED_EXPECTATIONS_CSV = PROCESSED_DIR / "categorized_expectations.csv"
CATEGORY_SUMMARY_CSV = PROCESSED_DIR / "category_summary.csv"

CHALLENGE_TEXT_COL = "pain_points"
EXPECTATION_TEXT_COL = "expectations"


## 1. Load processed records

In [ ]:
df_painpoints = pd.read_csv(CHALLENGES_CSV)
df_expectations = pd.read_csv(EXPECTATIONS_CSV)

print(f"Challenges: {len(df_painpoints)} | Expectations: {len(df_expectations)}")
print("Columns:", list(df_painpoints.columns))

# Merge NBD Kate → NBD Internal
df_painpoints.loc[df_painpoints["focus_group"] == "NBD Kate", "focus_group"] = "NBD Internal"
df_expectations.loc[df_expectations["focus_group"] == "NBD Kate", "focus_group"] = "NBD Internal"

print("Focus groups (pain):", df_painpoints["focus_group"].nunique())
print("Focus groups (exp):", df_expectations["focus_group"].nunique())
df_painpoints.head(3)


## 2. Sentiment

Uses `Thi144/sentiment-distilbert-7class` (7-way polarity), then maps to
`negative` / `neutral` / `positive` with source priors + keyword overrides.

| Fine label | Polarity |
|------------|----------|
| Very Negative … Slightly Negative | negative |
| Neutral | neutral |
| Slightly Positive … Very Positive | positive |


In [ ]:
classifier = build_sentiment_classifier()

examples = [
    "forms are a manual process",
    "Automate ticket routing between departments",
    "Hearings",
]
for example in examples:
    raw = classifier(example)
    # top_k=None → list[list[dict]]; otherwise list[dict]
    ranked = raw[0] if raw and isinstance(raw[0], list) else raw
    ranked = sorted(ranked, key=lambda r: r["score"], reverse=True)
    print(f'\n"{example}"')
    for item in ranked[:3]:
        meta = resolve_sentiment_label(item["label"])
        print(
            f"  {meta['label']:>8s}  {item['score']:.3f}  "
            f"← {meta['name']} ({item['label']})"
        )
    top = resolve_sentiment_label(ranked[0]["label"])
    print(f"→ polarity: {top['label']}  ({top['name']})")

print(f"\nModel: {SENTIMENT_MODEL}")
print("Classes:", {i: m["name"] for i, m in SENTIMENT_LABELS.items()})


In [ ]:
df_painpoints, df_expectations = run_sentiment(df_painpoints, df_expectations)

print("Pain sentiment:\n", df_painpoints["sentiment"].value_counts().to_string())
print("\nExpectation sentiment:\n", df_expectations["sentiment"].value_counts().to_string())

display(
    df_painpoints[["focus_group", "pain_points", "sentiment"]].sample(
        min(8, len(df_painpoints)), random_state=42
    )
)
display(
    df_expectations[["focus_group", "expectations", "sentiment"]].sample(
        min(8, len(df_expectations)), random_state=42
    )
)


In [ ]:
df_painpoints.to_csv(CHALLENGES_SCORED, index=False, encoding="utf-8-sig")
df_expectations.to_csv(EXPECTATIONS_SCORED, index=False, encoding="utf-8-sig")

print(f"Saved {len(df_painpoints)} → {CHALLENGES_SCORED}")
print(f"Saved {len(df_expectations)} → {EXPECTATIONS_SCORED}")


## 3. Realign misclassified rows

Move positive “pain” → expectations and negative “expectations” → pain points.


In [ ]:
n_pos = int((df_painpoints["sentiment"] == "positive").sum())
n_neg = int((df_expectations["sentiment"] == "negative").sum())

df_painpoints, df_expectations = realign_by_sentiment(df_painpoints, df_expectations)

print(f"Moved {n_pos} positive→expectations, {n_neg} negative→pain")
print(f"Shapes after realign: pain {df_painpoints.shape}, expectations {df_expectations.shape}")
print("Pain sentiment:\n", df_painpoints["sentiment"].value_counts().to_string())
print("\nExpectation sentiment:\n", df_expectations["sentiment"].value_counts().to_string())


## 4. Categorize records

Hybrid: title keywords → body keywords → semantic similarity.
When keyword and semantic agree, that label is preferred.


In [ ]:
print("Categories:", list(CATEGORY_DESCRIPTIONS.keys()))

df, expectations_df, challenge_embeddings, embedder = run_categorize(
    df_painpoints, df_expectations
)

print("\nChallenge categories:\n", df["Category"].value_counts().to_string())
print("\nMethods:\n", df["Category_Method"].value_counts().to_string())
print("\nExpectation categories:\n", expectations_df["Category"].value_counts().to_string())

display(
    df[["focus_group", CHALLENGE_TEXT_COL, "Category", "Category_Method", "Category_Confidence"]]
    .sample(min(10, len(df)), random_state=42)
)


## 5. Cluster challenges (UMAP + HDBSCAN)


In [ ]:
def assign_cluster_names(
    frame: pd.DataFrame,
    cluster_col: str = "Cluster",
    category_col: str = "Category",
    output_col: str = "Cluster_Label",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Name each cluster after its most common category label."""
    cluster_summary = (
        frame.groupby([cluster_col, category_col]).size().reset_index(name="Count")
    )
    dominant = (
        cluster_summary.sort_values("Count", ascending=False)
        .drop_duplicates(subset=[cluster_col])
        [[cluster_col, category_col]]
        .rename(columns={category_col: output_col})
    )
    return frame.merge(dominant, on=cluster_col, how="left"), cluster_summary


TARGET_CLUSTERS = len(CATEGORY_CONFIG)

reducer = umap.UMAP(
    n_neighbors=20, n_components=15, min_dist=0.0, metric="cosine", random_state=42
)
reduced = reducer.fit_transform(challenge_embeddings)

candidate_params = [
    (mcs, ms)
    for mcs in range(6, 13)
    for ms in sorted({1, 2, max(2, mcs // 2)})
]

sweep_rows, best_labels, best_params, best_score = [], None, None, -np.inf
for mcs, ms in candidate_params:
    labels = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric="euclidean",
        cluster_selection_method="eom",
    ).fit_predict(reduced)

    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    noise = int((labels == -1).sum())
    silhouette = (
        silhouette_score(reduced[mask], labels[mask], metric="euclidean")
        if mask.sum() > 1 and n_clusters > 1
        else -1.0
    )
    combined = (
        silhouette
        - abs(n_clusters - TARGET_CLUSTERS) * 0.02
        - noise / len(df) * 0.5
    )
    sweep_rows.append({
        "min_cluster_size": mcs,
        "min_samples": ms,
        "clusters": n_clusters,
        "noise": noise,
        "silhouette": round(float(silhouette), 4),
        "combined_score": round(float(combined), 4),
    })
    if combined > best_score:
        best_score, best_labels, best_params = combined, labels, (mcs, ms)

print("Best HDBSCAN params:", best_params, "score:", round(float(best_score), 4))
display(pd.DataFrame(sweep_rows).sort_values("combined_score", ascending=False).head(8))

df["Cluster"] = best_labels
df, cluster_summary = assign_cluster_names(df)
print("\nCluster → label counts:")
print(df.groupby(["Cluster", "Cluster_Label"]).size().to_string())


## 6. Save categorized outputs


In [ ]:
challenge_cols = [
    c for c in [
        "focus_group", CHALLENGE_TEXT_COL, "processed_text", "sentiment",
        "title", "Category", "Category_Method", "Category_Confidence",
        "Cluster", "Cluster_Label",
    ] if c in df.columns
]
expectation_cols = [
    c for c in [
        "focus_group", EXPECTATION_TEXT_COL, "processed_text", "sentiment",
        "title", "Category", "Category_Method", "Category_Confidence",
    ] if c in expectations_df.columns
]

df[challenge_cols].to_csv(CATEGORIZED_CHALLENGES_CSV, index=False, encoding="utf-8-sig")
expectations_df[expectation_cols].to_csv(
    CATEGORIZED_EXPECTATIONS_CSV, index=False, encoding="utf-8-sig"
)

category_summary = (
    df.groupby("Category", as_index=False).size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=False)
)
category_summary.to_csv(CATEGORY_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print(f"Saved {CATEGORIZED_CHALLENGES_CSV} ({len(df)} rows)")
print(f"Saved {CATEGORIZED_EXPECTATIONS_CSV} ({len(expectations_df)} rows)")
print(f"Saved {CATEGORY_SUMMARY_CSV}")
display(category_summary)
